In [1]:
import os
import json
import argparse
import orjson

In [2]:

# For the progress bar functionality.
# If you don't have it, install it with: pip install tqdm
try:
    from tqdm import tqdm
except ImportError:
    print("Warning: 'tqdm' library not found. Progress bar will not be shown.")
    print("You can install it with: pip install tqdm")
    # Create a dummy tqdm function if the library is not available
    def tqdm(iterable, *args, **kwargs):
        return iterable

def parse_boolean_string(s):
    """Converts a string 'True' or 'False' to a boolean value."""
    return s.strip().lower() == 'true'

def process_csv_file(file_path):
    """
    Reads a single CSV file, parses its four lines, and returns a dictionary
    in the specified format with sorted SNOMED codes.
    
    Args:
        file_path (str): The full path to the CSV file.
        
    Returns:
        dict: A dictionary containing the parsed data from the file,
              or None if the file is not valid.
    """
    try:
        with open(file_path, 'r') as f:
            lines = f.readlines()

        # Ensure the file has exactly 4 lines
        if len(lines) != 4:
            # This message can be noisy with tqdm, so it's commented out,
            # but can be re-enabled for debugging.
            # print(f"Warning: Skipping file {file_path}. Expected 4 lines, but found {len(lines)}.")
            return None

        # --- Data Extraction and Cleaning ---
        # Line 1: Exam ID
        exam_id_raw = lines[0].strip()
        exam_id = exam_id_raw
        
        # Line 2: Class IDs (comma-separated integers)
        class_ids = [int(cid) for cid in lines[1].strip().split(',')]
        
        # Line 3: Truth values (comma-separated booleans)
        truths = [parse_boolean_string(t) for t in lines[2].strip().split(',')]
        
        # Line 4: Probabilities (comma-separated floats)
        probabilities = [float(p) for p in lines[3].strip().split(',')]
        
        # --- Validation ---
        # Check if all lists have the same length
        if not (len(class_ids) == len(truths) == len(probabilities)):
            print(f"Warning: Skipping file {file_path}. Data length mismatch.")
            return None

        # --- Structuring the Data ---
        # Combine the data into a list of tuples for easy sorting
        combined_data = list(zip(class_ids, truths, probabilities))
        
        # Sort the data based on the class_id (the first element in each tuple)
        combined_data.sort(key=lambda x: x[0])
        
        # Create the snomed_vals dictionary from the sorted data
        snomed_vals = {}
        for class_id, truth, probability in combined_data:
            # JSON keys must be strings, so we convert the class_id
            snomed_vals[str(class_id)] = {
                "present": truth,
                "probability": probability
            }

        return {
            "exam_id": exam_id,
            "snomed_vals": snomed_vals
        }

    except (ValueError, IndexError):
        # Errors for individual files will not stop the whole process,
        # but the file will be skipped.
        print("error")
        return None
    except Exception:
        # Catch any other unexpected errors.
        return None


def convert_directory_to_json(input_dir, output_file, source):
    """
    Scans a directory for .csv files, processes them, and writes the
    combined result to a single JSON file, showing a progress bar.
    
    Args:
        input_dir (str): The path to the directory containing CSV files.
        output_file (str): The path for the output JSON file.
    """
    if not os.path.isdir(input_dir):
        print(f"Error: Input directory '{input_dir}' not found.")
        return

    all_data = []
    
    print(f"Scanning directory: {input_dir}")

    # Get a list of all csv files to process so tqdm knows the total
    csv_files_to_process = [f for f in os.listdir(input_dir) if f.lower().endswith('.csv')]
    
    # Iterate over the list with a tqdm progress bar
    for filename in tqdm(csv_files_to_process, desc="Processing files", unit="file"):
        file_path = os.path.join(input_dir, filename)
        file_data = process_csv_file(file_path)
        if file_data:
            # Add the source to the file_data dictionary
            file_data['source'] = source
            if source == 'ptb':
                file_data['chagas'] = False
            elif source == 'samitrop':
                file_data['chagas'] = True
            else:
                raise Exception("ptb or samitrop must be specified as source")
            all_data.append(file_data)

    # Write the compiled data to the output JSON file
    try:
        with open(output_file, 'w') as f:
            # Use indent for pretty-printing the JSON
            json.dump(all_data, f, indent=4)
        print(f"\nSuccess! Converted {len(all_data)} CSV files.")
        print(f"JSON output saved to: {output_file}")
    except Exception as e:
        print(f"\nError: Could not write to output file {output_file}: {e}")
    return all_data


In [7]:
SAMITROP_INPUT = "/sailhome/kelvinkn/scr2_juice/other_work/edwards/samitrop/prna_samitrop_outputs"
PTB_INPUT = "/sailhome/kelvinkn/scr2_juice/other_work/edwards/ptb/prna_ptb500_outputs"

In [8]:
samitrop_res = convert_directory_to_json(SAMITROP_INPUT, "samitrop_prna_outputs.json", "samitrop")

Scanning directory: /sailhome/kelvinkn/scr2_juice/other_work/edwards/samitrop/prna_samitrop_outputs


Processing files:   0%|          | 0/1631 [00:00<?, ?file/s]

Processing files: 100%|██████████| 1631/1631 [00:11<00:00, 142.85file/s]



Success! Converted 1631 CSV files.
JSON output saved to: samitrop_prna_outputs.json


In [13]:
ptb_res = convert_directory_to_json(PTB_INPUT, "ptb_prna_outputs.json", "ptb")

Scanning directory: /sailhome/kelvinkn/scr2_juice/other_work/edwards/ptb/prna_ptb500_outputs


Processing files:   0%|          | 0/21800 [00:00<?, ?file/s]

Processing files: 100%|██████████| 21800/21800 [00:18<00:00, 1171.50file/s]



Success! Converted 21800 CSV files.
JSON output saved to: ptb_prna_outputs.json


In [10]:
len(ptb_res)

21800

In [17]:
ptb_res[0]

{'exam_id': 17366,
 'snomed_vals': {'10370003': {'present': False, 'probability': 2.1883257e-05},
  '17338001': {'present': False, 'probability': 1.1481538e-05},
  '39732003': {'present': True, 'probability': 0.63365585},
  '47665007': {'present': False, 'probability': 4.3895398e-06},
  '59118001': {'present': False, 'probability': 4.509484e-06},
  '59931005': {'present': False, 'probability': 0.016080074},
  '63593006': {'present': False, 'probability': 0.0038303128},
  '111975006': {'present': False, 'probability': 0.004732529},
  '164889003': {'present': False, 'probability': 9.0332665e-05},
  '164890007': {'present': False, 'probability': 2.4648743e-06},
  '164909002': {'present': False, 'probability': 0.00010992688},
  '164917005': {'present': False, 'probability': 0.044810902},
  '164934002': {'present': True, 'probability': 0.120890565},
  '164947007': {'present': False, 'probability': 0.0001450436},
  '251146004': {'present': False, 'probability': 0.0027483173},
  '270492004': 

In [6]:
import orjson 
with open("/sailhome/kelvinkn/scr2_juice/code15_final_results.json", 'r') as f:
    code15 = orjson.loads(f.read())

In [14]:
# # 1. DEFINE YOUR PARAMETERS HERE
# # The file you already generated
# input_json_file = 'results.json' 
# # The new file you want to create with the added fields
# output_json_file = 'results_updated.json' 
# # The source string you want to use (e.g., "ptb")
# source_string = "ptb"

# print(f"Reading data from '{input_json_file}'...")

# # Read the existing JSON data
# try:
#     with open(input_json_file, 'r') as f:
#         data = json.load(f)
# except FileNotFoundError:
#     print(f"Error: Input file '{input_json_file}' not found. Please check the path.")
#     exit()
# except json.JSONDecodeError:
#     print(f"Error: Could not parse '{input_json_file}'. Make sure it is a valid JSON file.")
#     exit()

data = ptb_res
source_string = "ptb"
output_json_file = f"{source_string}_prna_outputs.json"

# Loop through each record in the list and add the new fields
for record in data:
    # Ensure the 'exam_id' key exists before using it
    if 'exam_id' in record:
        exam_id = record['exam_id']
        # Add the 'primary_id' field
        record['primary_id'] = f"{source_string}_{exam_id}"
        # # Add the 'source' field
        # record['source'] = source_string
    else:
        print(f"Warning: Skipping record because it is missing 'exam_id'. Record: {record}")


# Write the updated data to a new file
with open(output_json_file, 'w') as f:
    json.dump(data, f, indent=4)

print(f"Successfully updated {len(data)} records.")
print(f"New file saved to '{output_json_file}'.")


Successfully updated 21800 records.
New file saved to 'ptb_prna_outputs.json'.


In [17]:
samitrop_res[0]

{'exam_id': '267552',
 'snomed_vals': {'10370003': {'present': False, 'probability': 0.0023725384},
  '17338001': {'present': False, 'probability': 0.01693643},
  '39732003': {'present': False, 'probability': 0.0008683814},
  '47665007': {'present': False, 'probability': 0.0059894957},
  '59118001': {'present': False, 'probability': 0.042771954},
  '59931005': {'present': False, 'probability': 0.005784525},
  '63593006': {'present': False, 'probability': 0.0068113017},
  '111975006': {'present': False, 'probability': 0.0030670043},
  '164889003': {'present': True, 'probability': 0.11423636},
  '164890007': {'present': False, 'probability': 0.0011200496},
  '164909002': {'present': False, 'probability': 0.097825706},
  '164917005': {'present': False, 'probability': 0.0009337194},
  '164934002': {'present': False, 'probability': 0.005816611},
  '164947007': {'present': False, 'probability': 3.8718067e-06},
  '251146004': {'present': False, 'probability': 0.0025228267},
  '270492004': {'p

In [3]:
def get_json(path):
    with open(path, 'r') as f:
        return orjson.loads(f.read())

In [21]:
# samitrop = get_json("samitrop_prna_outputs.json")
# ptb = get_json("ptb_prna_outputs.json")
code15 = get_json("/sailhome/kelvinkn/scr2_juice/other_work/edwards/physionet2025/results/code15_prna_outputs.json")

In [22]:
# Convert all code15 exam_ids to strings
print("original length of code15:", len(code15))
num_code15_edited = 0
for record in tqdm(code15):
    if 'exam_id' in record:
        record['exam_id'] = str(record['exam_id'])
        num_code15_edited += 1
    else:
        print(f"Warning: Skipping record because it is missing 'exam_id'. Record: {record}")
print(f"Edited {num_code15_edited} records in code15 to have string exam_ids.")

original length of code15: 342530


100%|██████████| 342530/342530 [00:00<00:00, 666690.62it/s]

Edited 342530 records in code15 to have string exam_ids.


In [25]:
code15[0]['chagas']

False

In [4]:
samitrop = get_json("samitrop_prna_outputs.json")
ptb = get_json("ptb_prna_outputs.json")
code15 = get_json("results/code15_prna_outputs.json")

In [5]:
code15.extend(samitrop)
code15.extend(ptb)
len(code15)

365961

In [7]:
code15[-1]

{'exam_id': '10484_hr',
 'snomed_vals': {'10370003': {'present': False, 'probability': 2.4905735e-06},
  '17338001': {'present': False, 'probability': 1.9704686e-07},
  '39732003': {'present': False, 'probability': 0.0013435947},
  '47665007': {'present': False, 'probability': 0.00020648939},
  '59118001': {'present': False, 'probability': 1.536861e-07},
  '59931005': {'present': False, 'probability': 0.00010709988},
  '63593006': {'present': False, 'probability': 0.0012907604},
  '111975006': {'present': False, 'probability': 1.2141042e-05},
  '164889003': {'present': False, 'probability': 1.4959524e-05},
  '164890007': {'present': False, 'probability': 2.8857417e-07},
  '164909002': {'present': False, 'probability': 1.9465995e-08},
  '164917005': {'present': False, 'probability': 0.0049884743},
  '164934002': {'present': False, 'probability': 0.0020056076},
  '164947007': {'present': False, 'probability': 4.8973175e-06},
  '251146004': {'present': False, 'probability': 0.0013302428},

In [ ]:
json_bytes = orjson.dumps(code15, option=orjson.OPT_INDENT_2) # OPT_INDENT_2 for pretty-printing
json_string = json_bytes.decode('utf-8')
filename = "combined_prna_outputs.json"
# 3. Write the string to the file in text mode with UTF-8 encoding
with open(filename, 'w', encoding='utf-8') as f:
    f.write(json_string)

print(f"len {code15} records, JSON data successfully written to '{filename}'")


In [ ]:
with open(output_json_file, 'w') as f:
    json.dump(data, f, indent=4)

In [21]:
!ls -lh ptb_prna_outputs.json

-rw-r--r-- 1 kelvinkn users 70M Jun 12 02:19 ptb_prna_outputs.json


In [10]:
combined = get_json("combined_prna_outputs.json")

In [13]:
# Rename 'snowmed_vals' to 'snomed_vals'
values_changed = 0
for val in combined:
    # print(val)
    if 'snowmed_vals' in val:  # Check if the old key exists
        val['snomed_vals'] = val.pop('snowmed_vals')
        values_changed += 1
print(values_changed)

342530


In [6]:
# Samitrop conversion
samitrop_records_changed = 0
for record in combined:
    if record['source'] == "samitrop":
        record['chagas'] = True
        samitrop_records_changed += 1
print(samitrop_records_changed)

In [10]:
# ptb conversion
ptb_records_changed = 0
for record in combined:
    if record['source'] == "ptb":
        record['chagas'] = False
        ptb_records_changed += 1
print(ptb_records_changed)

21800


In [11]:
df = pd.DataFrame(combined)
df[df['source'] == 'ptb']

,exam_id,snowmed_vals,chagas,age,is_male,nn_predicted_age,1dAVb,RBBB,LBBB,SB,ST,AF,patient_id,death,timey,normal_ecg,trace_file,primary_id,source,snomed_vals
344161,17366,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ptb_17366,ptb,"{'10370003': {'present': False, 'probability':..."
344162,17541,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ptb_17541,ptb,"{'10370003': {'present': False, 'probability':..."
344163,3787,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ptb_3787,ptb,"{'10370003': {'present': False, 'probability':..."
344164,15009,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ptb_15009,ptb,"{'10370003': {'present': False, 'probability':..."
344165,3275,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ptb_3275,ptb,"{'10370003': {'present': False, 'probability':..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
365956,12870,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ptb_12870,ptb,"{'10370003': {'present': False, 'probability':..."
365957,12219,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ptb_12219,ptb,"{'10370003': {'present': False, 'probability':..."
365958,4065,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ptb_4065,ptb,"{'10370003': {'present': False, 'probability':..."
365959,4642,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ptb_4642,ptb,"{'10370003': {'present': False, 'probability':..."


In [23]:
def dump_json(data, filename):
    """
    Serializes Python data to JSON bytes using orjson and writes it to a file.
    This method is generally faster for large datasets due to orjson's
    performance and direct byte output.

    Args:
        data: The Python object (e.g., dict, list) to serialize.
        filename (str): The name of the file to write to.
    """
    # orjson.dumps returns bytes directly, which is efficient for file I/O.
    # No need for .encode('utf-8') as orjson already handles the encoding.
    # Added orjson.OPT_INDENT_2 to enable 2-space indentation for human readability.
    json_bytes = orjson.dumps(data, option=orjson.OPT_INDENT_2)

    # Open the file in binary write mode ('wb')
    with open(filename, 'wb') as f:
        f.write(json_bytes)

In [24]:
dump_json(combined, "test_combined_prna_outputs.json")

In [17]:
len(combined)

365961

In [21]:
exam_ids = set()
for val in combined:
    exam_id = val['primary_id']
    if exam_id in exam_ids:
        print(exam_id)
    exam_ids.add(exam_id)

ptb_1
